# Tutorial 1 — SCF and Orbital Localization

This notebook covers the two preparatory steps that precede any AEGISS active space
selection: obtaining a mean-field reference wavefunction and (optionally) transforming
the canonical molecular orbitals (MOs) into a localized representation.

**Why localize?**  
Canonical HF/KS orbitals are delocalized over the whole molecule by symmetry.  
Localized orbitals concentrate electron density on chemically relevant fragments,
making the subsequent DMRG entropy and AO-projection screening more physically
meaningful.

**Contents**
1. Build the molecule
2. Run RHF and RKS
3. Boys localization
4. Pipek–Mezey (PM) localization
5. Edmiston–Ruedenberg (ER) localization
6. Intrinsic Bond Orbitals (IBOs)
7. Intrinsic Atomic Orbitals (IAOs)
8. Natural Orbitals (NOs) from MP2
9. Frozen Natural Orbitals (FNOs)

## 1. Imports

In [ ]:
import numpy as np

from aegiss.initial_guess.scf import build_mol, run_scf
from aegiss.localization.orbital_localization import localize

## 2. Build the molecule

`build_mol` is a thin wrapper around `pyscf.gto.Mole`.  Any keyword accepted by
`Mole` (e.g. `basis`, `charge`, `spin`, `symmetry`) can be passed as keyword arguments.

In [ ]:
# Benzene – D6h, planar, 6 pi electrons in 6 pi orbitals
benzene_geom = """
C   1.40272  0.00000  0.00000
C   0.70136  1.21451  0.00000
C  -0.70136  1.21451  0.00000
C  -1.40272  0.00000  0.00000
C  -0.70136 -1.21451  0.00000
C   0.70136 -1.21451  0.00000
H   2.49118  0.00000  0.00000
H   1.24559  2.15720  0.00000
H  -1.24559  2.15720  0.00000
H  -2.49118  0.00000  0.00000
H  -1.24559 -2.15720  0.00000
H   1.24559 -2.15720  0.00000
"""

mol = build_mol(
    atom=benzene_geom,
    basis="sto-3g",   # small basis for speed; use cc-pVDZ for production
    symmetry=False,
    verbose=0,
)
print(f"Number of atoms  : {mol.natm}")
print(f"Number of AOs    : {mol.nao}")
print(f"Number of electrons: {mol.nelectron}")

## 3. Run RHF

`run_scf` dispatches to PySCF's `RHF`, `UHF`, `RKS`, or `UKS` depending on the
`method` argument.  It calls `kernel()` and returns the converged mean-field object.

In [ ]:
mf = run_scf(mol, method="RHF")

print(f"RHF energy       : {mf.e_tot:.8f} Ha")
print(f"Number of MOs    : {mf.mo_coeff.shape[1]}")
print(f"Occupied MOs     : {int(np.sum(mf.mo_occ > 0))}")

In [ ]:
# DFT alternative — uncomment to compare energies
# mf_dft = run_scf(mol, method="RKS", xc="b3lyp")
# print(f"B3LYP energy : {mf_dft.e_tot:.8f} Ha")

## 3b. Scalar relativistic correction (X2C)

For molecules containing heavier elements — transition metals, lanthanides, actinides
— scalar relativistic effects contract the core s and p shells and alter valence
orbital energies noticeably. AEGISS exposes PySCF's exact two-component (X2C)
transformation via the `relativistic` parameter of `run_scf`.

X2C folds the mass-velocity and Darwin corrections into a modified one-electron
Hamiltonian. The rest of the workflow (DMRG integral fetch via `CASCI.get_h1eff`,
entropy screening, AO projection) requires no changes — the relativistic integrals
propagate automatically through the `mf` object.

> **Basis sets**: for production calculations on heavy elements use a relativistically
> contracted basis (e.g. `ano-rcc`, `dyall-vdz`, `x2c-svpall`) together with
> `relativistic='x2c'`. The STO-3G basis used in this tutorial is not optimised for
> relativistic corrections; the example below illustrates the API only.

In [ ]:
# X2C scalar relativistic RHF
# The correction is negligible for C/H but becomes significant for transition
# metals (e.g. Fe in Ferrocene) and heavier systems.
mf_x2c = run_scf(mol, method="RHF", relativistic="x2c")

print(f"X2C-RHF energy    : {mf_x2c.e_tot:.8f} Ha")
print(f"Non-rel RHF energy: {mf.e_tot:.8f} Ha")
print(f"Relativistic correction: {mf_x2c.e_tot - mf.e_tot:.6f} Ha")

# X2C composes with DFT and solvent models
# mf_x2c_dft = run_scf(mol, method="RKS", xc="b3lyp", relativistic="x2c")
# mf_x2c_pcm = run_scf(mol, method="RHF", relativistic="x2c", solvent_info=("PCM", 78.4))

## 4. Boys localization

Boys localization minimises the spatial spread $\langle r^2 \rangle - \langle r \rangle^2$
of each orbital, producing chemically intuitive bond and lone-pair orbitals.

By default only the occupied space is localized.  Pass `on_virtual=True` to also
localize the virtual space — useful before entropy screening.

In [ ]:
mo_boys = localize("boys", mol, mf)

# Use the localized MOs in a *copy* of the mean-field object, so `mf` keeps
# the canonical orbitals for the comparison below and the next sections
mf_boys = mf.copy()
mf_boys.mo_coeff = mo_boys

print(f"MO shape after Boys localization: {mo_boys.shape}")
print("Localized occupied MOs differ from canonical:")
print(f"  Max |diff| (occ block): {np.max(np.abs(mo_boys[:, :21] - mf.mo_coeff[:, :21])):.4f}")

In [ ]:
# Localize both occupied and virtual spaces
mo_boys_full = localize("boys", mol, mf, on_virtual=True)
print(f"Full MO shape: {mo_boys_full.shape}")

## 5. Pipek–Mezey (PM) localization

PM localization maximises the sum of squared Mulliken partial charges on atoms,
tending to preserve σ/π separation and produce orbitals tied to specific atoms.
It is generally preferred over Boys for systems with distinct bond types.

In [ ]:
mo_pm = localize("pm", mol, mf)
print(f"MO shape after PM localization : {mo_pm.shape}")

# PM preserves sigma/pi separation in benzene — inspect orbital norms
n_occ = int(np.sum(mf.mo_occ > 0))
print(f"\nNorm of occ block (should be unitary): {np.linalg.norm(mo_pm[:, :n_occ].T @ mol.intor('int1e_ovlp') @ mo_pm[:, :n_occ] - np.eye(n_occ)):.2e}")

## 6. Edmiston–Ruedenberg (ER) localization

ER localization maximises the orbital self-repulsion energy $\langle ii | ii \rangle$.
It is the most rigorous localization scheme but also the most expensive due to
two-electron integral transformations.  For large systems prefer Boys or PM.

In [ ]:
# 'er' and 'edmiston' are aliases for the same method
mo_er = localize("er", mol, mf)
print(f"MO shape after ER localization : {mo_er.shape}")

## 7. Intrinsic Bond Orbitals (IBOs)

IBOs (Knizia, JCTC 2013) localise occupied MOs by maximising their population on
**Intrinsic Atomic Orbitals** (IAOs). They tend to give chemically intuitive
bond / lone-pair orbitals, preserve σ/π separation similarly to PM, and are
especially well-suited for systems with mixed ionic/covalent character and for
transition metals.

IBOs operate only on the occupied space; the virtual block is kept unchanged.

In [ ]:
mo_ibos = localize("ibos", mol, mf)
print(f"MO shape after IBO localization: {mo_ibos.shape}")

n_occ = int(np.sum(mf.mo_occ > 0))
s = mol.intor("int1e_ovlp")
overlap = mo_ibos[:, :n_occ].T @ s @ mo_ibos[:, :n_occ]
print(f"Occupied block orthonormality error: {np.max(np.abs(overlap - np.eye(n_occ))):.2e}")

## 8. Intrinsic Atomic Orbitals (IAOs)

IAOs are a set of orthonormal orbitals expressed in the AO basis that closely resemble
the functions of a reference minimal basis (default: `"minao"`). They span the occupied
MO space and provide a chemically meaningful atom-centred representation.

IAOs are typically used as:
- a localized occupied basis for population / charge analysis
- the underlying population measure inside the IBO construction

There is one IAO per function of the minimal basis, which is in general more than
the number of occupied MOs, so `localize("iaos")` returns the IAO basis itself,
of shape `(nao, n_iao)`, not a replacement for `mf.mo_coeff`.

In [ ]:
iaos = localize("iaos", mol, mf)
print(f"IAO basis shape: {iaos.shape}  ({n_occ} occupied MOs)")

overlap = iaos.T @ s @ iaos
print(f"IAO orthonormality error: {np.max(np.abs(overlap - np.eye(iaos.shape[1]))):.2e}")

## 9. Natural Orbitals (NOs)

Natural orbitals diagonalise the one-particle reduced density matrix (1-RDM) of a
correlated wavefunction.  They are ordered by occupation number, concentrating the
correlation description into the fewest possible orbitals — ideal as an AEGISS
initial guess.

NOs always use the full orbital space. To freeze core orbitals use FNOs instead.

In [ ]:
# NOs from MP2 — fast and reliable for weakly correlated systems
mo_nos = localize("nos", mf, correlated_method="mp2")
print(f"MO shape after MP2 NOs: {mo_nos.shape}")

In [ ]:
# Split NOs: diagonalise occupied and virtual blocks independently
# Useful when you want to keep canonical structure within each space
mo_nos_split = localize("nos", mf, correlated_method="mp2", split=True)
print(f"MO shape after split MP2 NOs: {mo_nos_split.shape}")

In [ ]:
mo_nos = localize("nos", mf, correlated_method="mp2")
print(f"MO shape after MP2 NOs: {mo_nos.shape}")

# NOs always use the full orbital space.
# To freeze core orbitals use FNOs (see section 10).

## 10. Frozen Natural Orbitals (FNOs)

FNOs partition the orbital space into a frozen core and an active set before
performing the NO transformation.  This reduces the cost for larger systems by
excluding deep core orbitals from the correlated calculation.

Two keyword arguments control the partition — use whichever is more natural:

| Argument | Meaning |
|---|---|
| `frozen=list[int]` | Indices of occupied orbitals to **exclude** from correlation (recommended) |
| `active=list[int]` | Indices of occupied orbitals to **include** in correlation (complement view) |
| `active=True` | Include **all** occupied orbitals — equivalent to standard NOs (default) |

When `frozen` is provided it takes precedence over `active`.

In [ ]:
# FNOs with all occupied orbitals active (equivalent to standard NOs here)
mo_fnos = localize("fnos", mf, correlated_method="mp2", active=True)
print(f"MO shape after FNOs: {mo_fnos.shape}")

In [ ]:
# Freeze the 6 carbon 1s orbitals (core) and build FNOs over the valence space.
n_core = 6
mo_fnos_frozen = localize("fnos", mf, correlated_method="mp2", frozen=list(range(n_core)))
print(f"MO shape after FNOs (frozen=[0..{n_core-1}]): {mo_fnos_frozen.shape}")

# Equivalent using active= (the complement of the frozen set)
nocc = int(np.sum(mf.mo_occ > 0))
mo_fnos_active = localize("fnos", mf, correlated_method="mp2", active=list(range(n_core, nocc)))
print(f"MO shape after FNOs (active=[{n_core}..{nocc-1}]): {mo_fnos_active.shape}")

## Summary

| Method | `localize` key | Criterion | Typical use |
|---|---|---|---|
| Boys | `"boys"` | Minimum orbital spread | General purpose, π systems |
| Pipek–Mezey | `"pm"` | Maximum atomic charge | σ/π separation, transition metals |
| Edmiston–Ruedenberg | `"er"` / `"edmiston"` | Maximum self-repulsion | Rigorous, expensive |
| IBOs | `"ibos"` / `"ibo"` | Maximum IAO population | Transition metals, mixed bonding |
| IAOs | `"iaos"` / `"iao"` | Minimal-basis projection | Population analysis, IBO input |
| Natural Orbitals | `"nos"` | Diagonalise full 1-RDM | Pre-CASSCF guess |
| Frozen NOs | `"fnos"` | Diagonalise virtual block of 1-RDM | Active-space truncation |
| FNOs (frozen core) | `"fnos"` + `frozen=` | Same, core excluded | Heavy atoms, large systems |

**Frozen orbitals** (`frozen=list[int]`) can be passed to `"fnos"`.
The listed orbitals are excluded from the correlated calculation but kept in the
output, so `mf.mo_coeff` retains its full shape and the rest of the AEGISS pipeline
requires no changes.

The localized `mo_coeff` can be passed directly to any subsequent AEGISS step by
replacing `mf.mo_coeff` before calling `select_aegiss` or `select_by_ao_projection`.

**Next**: [02_dmrg.ipynb](02_dmrg.ipynb) — running the DMRG calculation and inspecting orbital entropies.